#Part 1 – Environment Setup

In [1]:
!pip install -q pyspark

In [2]:
import sys

print("Python Version:", sys.version)

Python Version: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]


In [3]:
!java -version

openjdk version "17.0.19" 2026-04-21
OpenJDK Runtime Environment (build 17.0.19+10-1-22.04.2-Ubuntu)
OpenJDK 64-Bit Server VM (build 17.0.19+10-1-22.04.2-Ubuntu, mixed mode, sharing)


In [10]:
import pyspark
from pyspark.sql import SparkSession

print("PySpark Version:", pyspark.__version__)

PySpark Version: 4.0.3


In [11]:
spark_session = SparkSession.builder \
    .appName("NYC Taxi Trip Analytics") \
    .master("local[*]") \
    .getOrCreate()

print("Spark Session Created Successfully!")
print("Spark Version:", spark_session.version)
print("Master:", spark_session.sparkContext.master)

Spark Session Created Successfully!
Spark Version: 4.0.3
Master: local[*]


In [12]:
import platform
import sys

print("Python Version:", sys.version)
print("Operating System:", platform.system())
print("Spark Version:", spark_session.version)

Python Version: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Operating System: Linux
Spark Version: 4.0.3


In [13]:
print("========== SPARK CONFIGURATION ==========")

for key, value in spark_session.sparkContext.getConf().getAll():
    print(f"{key}: {value}")

========== SPARK CONFIGURATION ==========
spark.rdd.compress: True
spark.hadoop.fs.s3a.vectored.read.min.seek.size: 128K
spark.driver.port: 41451
spark.executor.extraJavaOptions: -Djava.net.preferIPv6Addresses=false -XX:+IgnoreUnrecognizedVMOptions --add-modules=jdk.incubator.vector --add-opens=java.base/java.lang=ALL-UNNAMED --add-opens=java.base/java.lang.invoke=ALL-UNNAMED --add-opens=java.base/java.lang.reflect=ALL-UNNAMED --add-opens=java.base/java.io=ALL-UNNAMED --add-opens=java.base/java.net=ALL-UNNAMED --add-opens=java.base/java.nio=ALL-UNNAMED --add-opens=java.base/java.util=ALL-UNNAMED --add-opens=java.base/java.util.concurrent=ALL-UNNAMED --add-opens=java.base/java.util.concurrent.atomic=ALL-UNNAMED --add-opens=java.base/jdk.internal.ref=ALL-UNNAMED --add-opens=java.base/sun.nio.ch=ALL-UNNAMED --add-opens=java.base/sun.nio.cs=ALL-UNNAMED --add-opens=java.base/sun.security.action=ALL-UNNAMED --add-opens=java.base/sun.util.calendar=ALL-UNNAMED --add-opens=java.security.jgss/su

#PART 2 — LOAD DATASET

In [14]:
taxi_df = spark_session.read.parquet(
    "yellow_tripdata_2024-01.parquet"
)

print("Dataset Loaded Successfully!")

Dataset Loaded Successfully!


In [15]:
taxi_df.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)



In [16]:
print("Number of Records:", taxi_df.count())

Number of Records: 2964624


In [17]:
print("Number of Columns:", len(taxi_df.columns))

Number of Columns: 19


In [18]:
taxi_df.show(20, truncate=False)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|2       |2024-01-01 00:57:55 |2024-01-01 01:17:43  |1              |1.72         |1         |N                 |186         |79          |2           |17.7       |1.0  |0.5    |0.0      

PART 3 — EDA

In [19]:
total_trips = taxi_df.count()

print("Total Trips:", total_trips)

Total Trips: 2964624


In [20]:
from pyspark.sql.functions import min

earliest_trip = taxi_df.select(
    min("tpep_pickup_datetime").alias("Earliest_Trip_Date")
)

earliest_trip.show()

+-------------------+
| Earliest_Trip_Date|
+-------------------+
|2002-12-31 22:59:39|
+-------------------+



In [21]:
from pyspark.sql.functions import max

latest_trip = taxi_df.select(
    max("tpep_pickup_datetime").alias("Latest_Trip_Date")
)

latest_trip.show()

+-------------------+
|   Latest_Trip_Date|
+-------------------+
|2024-02-01 00:01:15|
+-------------------+



In [22]:
unique_vendors_df = taxi_df.select(
    "VendorID"
).distinct()

unique_vendors_df.show()

print(
    "Number of Unique Vendors:",
    unique_vendors_df.count()
)

+--------+
|VendorID|
+--------+
|       1|
|       2|
|       6|
+--------+

Number of Unique Vendors: 3


In [23]:
from pyspark.sql.functions import avg

average_distance_df = taxi_df.select(
    avg("trip_distance").alias("Average_Trip_Distance")
)

average_distance_df.show()

+---------------------+
|Average_Trip_Distance|
+---------------------+
|   3.6521691789580624|
+---------------------+



In [24]:
average_fare_df = taxi_df.select(
    avg("fare_amount").alias("Average_Fare")
)

average_fare_df.show()

+------------------+
|      Average_Fare|
+------------------+
|18.175061916792536|
+------------------+



In [25]:
maximum_fare_df = taxi_df.select(
    max("fare_amount").alias("Maximum_Fare")
)

maximum_fare_df.show()

+------------+
|Maximum_Fare|
+------------+
|      5000.0|
+------------+



In [26]:
minimum_fare_df = taxi_df.select(
    min("fare_amount").alias("Minimum_Fare")
)

minimum_fare_df.show()

+------------+
|Minimum_Fare|
+------------+
|      -899.0|
+------------+



In [27]:
average_passenger_df = taxi_df.select(
    avg("passenger_count").alias("Average_Passenger_Count")
)

average_passenger_df.show()

+-----------------------+
|Average_Passenger_Count|
+-----------------------+
|     1.3392808966805005|
+-----------------------+



In [28]:
payment_methods_df = taxi_df.select(
    "payment_type"
).distinct()

payment_methods_df.show()

print(
    "Number of Payment Methods:",
    payment_methods_df.count()
)

+------------+
|payment_type|
+------------+
|           1|
|           3|
|           2|
|           4|
|           0|
+------------+

Number of Payment Methods: 5


#PART 4 — DATA CLEANING

In [29]:
print(
    "Original Number of Records:",
    taxi_df.count()
)

Original Number of Records: 2964624


In [30]:
clean_taxi_df = taxi_df.dropDuplicates()

print(
    "Records After Removing Duplicates:",
    clean_taxi_df.count()
)

Records After Removing Duplicates: 2964624


In [31]:
clean_taxi_df = clean_taxi_df.filter(
    clean_taxi_df.trip_distance > 0
)

print(
    "Records After Removing Invalid Trip Distances:",
    clean_taxi_df.count()
)

Records After Removing Invalid Trip Distances: 2904253


In [32]:
clean_taxi_df = clean_taxi_df.filter(
    clean_taxi_df.fare_amount >= 0
)

print(
    "Records After Removing Negative Fares:",
    clean_taxi_df.count()
)

Records After Removing Negative Fares: 2870188


In [33]:
from pyspark.sql.functions import col, when, count

missing_values_df = clean_taxi_df.select([
    count(
        when(col(column_name).isNull(), column_name)
    ).alias(column_name)
    for column_name in clean_taxi_df.columns
])

missing_values_df.show(vertical=True)

-RECORD 0-----------------------
 VendorID              | 0      
 tpep_pickup_datetime  | 0      
 tpep_dropoff_datetime | 0      
 passenger_count       | 115293 
 trip_distance         | 0      
 RatecodeID            | 115293 
 store_and_fwd_flag    | 115293 
 PULocationID          | 0      
 DOLocationID          | 0      
 payment_type          | 0      
 fare_amount           | 0      
 extra                 | 0      
 mta_tax               | 0      
 tip_amount            | 0      
 tolls_amount          | 0      
 improvement_surcharge | 0      
 total_amount          | 0      
 congestion_surcharge  | 115293 
 Airport_fee           | 115293 



In [34]:
clean_taxi_df = clean_taxi_df.dropna()

print(
    "Records After Handling Missing Values:",
    clean_taxi_df.count()
)

Records After Handling Missing Values: 2754895


In [35]:
print(
    "Final Number of Records:",
    clean_taxi_df.count()
)

clean_taxi_df.show(10, truncate=False)

Final Number of Records: 2754895
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|2       |2024-01-01 00:47:21 |2024-01-01 00:51:58  |2              |1.13         |1         |N                 |238         |166         |1           |7.

#PART 5 — SPARK TRANSFORMATIONS


In [36]:
high_fare_df = clean_taxi_df.filter(
    clean_taxi_df.fare_amount > 50
)

high_fare_df.show(10)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|       2| 2024-01-01 01:41:09|  2024-01-01 02:07:52|              1|        19.69|         5|                 N|         262|         265|           1|      100.0|  0.0|    0.0|      20.

In [37]:
selected_columns_df = clean_taxi_df.select(
    "VendorID",
    "passenger_count",
    "trip_distance",
    "fare_amount"
)

selected_columns_df.show(10)

+--------+---------------+-------------+-----------+
|VendorID|passenger_count|trip_distance|fare_amount|
+--------+---------------+-------------+-----------+
|       2|              2|         1.13|        7.9|
|       1|              3|          0.8|        6.5|
|       2|              1|         4.43|       31.0|
|       2|              1|         3.64|       17.7|
|       2|              2|          5.9|       30.3|
|       2|              2|         3.43|       24.7|
|       2|              1|         0.95|        8.6|
|       2|              2|         1.83|       10.7|
|       2|              1|         2.25|       14.2|
|       1|              2|          0.8|        7.2|
+--------+---------------+-------------+-----------+
only showing top 10 rows


In [38]:
from pyspark.sql.functions import round

fare_per_mile_df = clean_taxi_df.withColumn(
    "fare_per_mile",
    round(
        clean_taxi_df.fare_amount /
        clean_taxi_df.trip_distance,
        2
    )
)

fare_per_mile_df.select(
    "trip_distance",
    "fare_amount",
    "fare_per_mile"
).show(10)

+-------------+-----------+-------------+
|trip_distance|fare_amount|fare_per_mile|
+-------------+-----------+-------------+
|         1.13|        7.9|         6.99|
|          0.8|        6.5|         8.13|
|         4.43|       31.0|          7.0|
|         3.64|       17.7|         4.86|
|          5.9|       30.3|         5.14|
|         3.43|       24.7|          7.2|
|         0.95|        8.6|         9.05|
|         1.83|       10.7|         5.85|
|         2.25|       14.2|         6.31|
|          0.8|        7.2|          9.0|
+-------------+-----------+-------------+
only showing top 10 rows


In [39]:
sorted_fare_df = clean_taxi_df.orderBy(
    clean_taxi_df.fare_amount.desc()
)

sorted_fare_df.show(10)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|       2| 2024-01-14 10:08:11|  2024-01-16 13:54:22|              1|        31.95|         1|                 N|         220|         220|           2|     2221.3|  0.0|    0.5|       0.

In [40]:
dropped_column_df = clean_taxi_df.drop(
    "store_and_fwd_flag"
)

dropped_column_df.show(10)

+--------+--------------------+---------------------+---------------+-------------+----------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|       2| 2024-01-01 00:47:21|  2024-01-01 00:51:58|              2|         1.13|         1|         238|         166|           1|        7.9|  1.0|    0.5|      2.58|         0.0|                  1.0|       15.48|                 2.5|     

In [41]:
unique_payment_df = clean_taxi_df.select(
    "payment_type"
).distinct()

unique_payment_df.show()

+------------+
|payment_type|
+------------+
|           1|
|           3|
|           2|
|           4|
+------------+



In [42]:
from pyspark.sql.functions import avg

payment_summary_df = clean_taxi_df.groupBy(
    "payment_type"
).agg(
    avg("fare_amount").alias("Average_Fare")
)

payment_summary_df.show()

+------------+------------------+
|payment_type|      Average_Fare|
+------------+------------------+
|           1|18.376087258999238|
|           3|17.040781294018178|
|           2|18.613623047913922|
|           4| 19.66531404344601|
+------------+------------------+



In [43]:
payment_type_df = clean_taxi_df.select(
    "payment_type"
).distinct()
payment_joined_df = payment_type_df.join(
    payment_summary_df,
    on="payment_type",
    how="inner"
)

payment_joined_df.show()

+------------+------------------+
|payment_type|      Average_Fare|
+------------+------------------+
|           1|18.376087258999238|
|           3|17.040781294018178|
|           2|18.613623047913922|
|           4| 19.66531404344601|
+------------+------------------+



In [46]:
aliased_columns_df = clean_taxi_df.select(
    clean_taxi_df.VendorID.alias("Vendor"),
    clean_taxi_df.trip_distance.alias("Distance"),
    clean_taxi_df.fare_amount.alias("Fare")
)

aliased_columns_df.show(10)

+------+--------+----+
|Vendor|Distance|Fare|
+------+--------+----+
|     2|    1.13| 7.9|
|     1|     0.8| 6.5|
|     2|    4.43|31.0|
|     2|    3.64|17.7|
|     2|     5.9|30.3|
|     2|    3.43|24.7|
|     2|    0.95| 8.6|
|     2|    1.83|10.7|
|     2|    2.25|14.2|
|     1|     0.8| 7.2|
+------+--------+----+
only showing top 10 rows


In [45]:
repartitioned_taxi_df = clean_taxi_df.repartition(4)

print(
    "Number of Partitions:",
    repartitioned_taxi_df.rdd.getNumPartitions()
)

Number of Partitions: 4


#PART 6 — SPARK SQL

In [47]:
clean_taxi_df.createOrReplaceTempView(
    "nyc_taxi_trips"
)

print("Temporary SQL View Created Successfully!")

Temporary SQL View Created Successfully!


In [48]:
sql_result_1 = spark_session.sql("""
SELECT
    trip_distance,
    fare_amount,
    passenger_count
FROM nyc_taxi_trips
ORDER BY trip_distance DESC
LIMIT 10
""")

sql_result_1.show()

+-------------+-----------+---------------+
|trip_distance|fare_amount|passenger_count|
+-------------+-----------+---------------+
|     15400.32|       28.9|              1|
|     10879.28|       70.0|              1|
|      1715.22|       70.0|              1|
|        971.8|       21.5|              1|
|        964.6|       39.5|              1|
|        277.4|       33.8|              2|
|       246.22|        8.6|              1|
|       233.25|     1616.5|              1|
|       210.82|      500.0|              1|
|        210.2|      650.0|              1|
+-------------+-----------+---------------+



In [49]:
sql_result_2 = spark_session.sql("""
SELECT
    PULocationID,
    COUNT(*) AS Total_Trips
FROM nyc_taxi_trips
GROUP BY PULocationID
ORDER BY Total_Trips DESC
LIMIT 10
""")

sql_result_2.show()

+------------+-----------+
|PULocationID|Total_Trips|
+------------+-----------+
|         132|     138130|
|         237|     137093|
|         161|     136500|
|         236|     129604|
|         162|     102308|
|         186|     100737|
|         230|     100008|
|         142|      98906|
|         138|      87253|
|         239|      82391|
+------------+-----------+



In [50]:
sql_result_3 = spark_session.sql("""
SELECT
    payment_type,
    ROUND(AVG(fare_amount), 2) AS Average_Fare
FROM nyc_taxi_trips
GROUP BY payment_type
ORDER BY payment_type
""")

sql_result_3.show()

+------------+------------+
|payment_type|Average_Fare|
+------------+------------+
|           1|       18.38|
|           2|       18.61|
|           3|       17.04|
|           4|       19.67|
+------------+------------+



In [51]:
sql_result_4 = spark_session.sql("""
SELECT
    HOUR(tpep_pickup_datetime) AS Pickup_Hour,
    COUNT(*) AS Total_Trips
FROM nyc_taxi_trips
GROUP BY Pickup_Hour
ORDER BY Total_Trips DESC
LIMIT 1
""")

sql_result_4.show()

+-----------+-----------+
|Pickup_Hour|Total_Trips|
+-----------+-----------+
|         18|     198037|
+-----------+-----------+



In [53]:
sql_result_5 = spark_session.sql("""
SELECT
    VendorID,
    trip_distance,
    fare_amount
FROM nyc_taxi_trips
WHERE trip_distance > 20
ORDER BY trip_distance DESC
""")

sql_result_5.show(10)

+--------+-------------+-----------+
|VendorID|trip_distance|fare_amount|
+--------+-------------+-----------+
|       2|     15400.32|       28.9|
|       2|     10879.28|       70.0|
|       2|      1715.22|       70.0|
|       1|        971.8|       21.5|
|       1|        964.6|       39.5|
|       2|        277.4|       33.8|
|       2|       246.22|        8.6|
|       2|       233.25|     1616.5|
|       2|       210.82|      500.0|
|       1|        210.2|      650.0|
+--------+-------------+-----------+
only showing top 10 rows


In [54]:
sql_result_6 = spark_session.sql("""
SELECT
    MONTH(tpep_pickup_datetime) AS Month,
    ROUND(SUM(total_amount), 2) AS Revenue
FROM nyc_taxi_trips
GROUP BY Month
ORDER BY Month
""")

sql_result_6.show()

+-----+-------------+
|Month|      Revenue|
+-----+-------------+
|    1|7.542614842E7|
|    2|        90.47|
|   12|       235.12|
+-----+-------------+



In [55]:
sql_result_7 = spark_session.sql("""
SELECT
    VendorID,
    ROUND(AVG(trip_distance), 2) AS Average_Distance
FROM nyc_taxi_trips
GROUP BY VendorID
""")

sql_result_7.show()

+--------+----------------+
|VendorID|Average_Distance|
+--------+----------------+
|       1|            3.13|
|       2|            3.35|
+--------+----------------+



In [56]:
sql_result_8 = spark_session.sql("""
SELECT
    VendorID,
    fare_amount
FROM nyc_taxi_trips
ORDER BY fare_amount DESC
LIMIT 10
""")

sql_result_8.show()

+--------+-----------+
|VendorID|fare_amount|
+--------+-----------+
|       2|     2221.3|
|       2|     1616.5|
|       2|      912.3|
|       2|      899.0|
|       2|      820.0|
|       2|      761.1|
|       2|      749.2|
|       2|      744.3|
|       2|      739.4|
|       2|      700.0|
+--------+-----------+



In [57]:
sql_result_9 = spark_session.sql("""
SELECT
    ROUND(AVG(passenger_count), 2) AS Average_Passengers
FROM nyc_taxi_trips
""")

sql_result_9.show()

+------------------+
|Average_Passengers|
+------------------+
|              1.34|
+------------------+



In [58]:
sql_result_10 = spark_session.sql("""
SELECT
    payment_type,
    COUNT(*) AS Total_Trips
FROM nyc_taxi_trips
GROUP BY payment_type
ORDER BY Total_Trips DESC
""")

sql_result_10.show()

+------------+-----------+
|payment_type|Total_Trips|
+------------+-----------+
|           1|    2298422|
|           2|     422945|
|           4|      22879|
|           3|      10649|
+------------+-----------+



In [59]:
output_query_1 = spark_session.sql("""
SELECT
    payment_type,
    COUNT(*) AS Total_Trips
FROM nyc_taxi_trips
GROUP BY payment_type
""")

output_query_1.toPandas().to_csv(
    "query1.csv",
    index=False
)

print("query1.csv created successfully!")

query1.csv created successfully!


In [60]:
output_query_2 = spark_session.sql("""
SELECT
    VendorID,
    AVG(fare_amount) AS Average_Fare
FROM nyc_taxi_trips
GROUP BY VendorID
""")

output_query_2.toPandas().to_csv(
    "query2.csv",
    index=False
)

print("query2.csv created successfully!")

query2.csv created successfully!


In [61]:
output_query_3 = spark_session.sql("""
SELECT
    PULocationID,
    COUNT(*) AS Trips
FROM nyc_taxi_trips
GROUP BY PULocationID
ORDER BY Trips DESC
LIMIT 10
""")

output_query_3.toPandas().to_csv(
    "query3.csv",
    index=False
)

print("query3.csv created successfully!")

query3.csv created successfully!


#PART 7 — WINDOW FUNCTIONS

In [62]:
from pyspark.sql.window import Window
from pyspark.sql.functions import (
    row_number,
    rank,
    dense_rank
)

In [63]:
fare_window = Window.orderBy(
    clean_taxi_df.fare_amount.desc()
)

In [64]:
row_number_df = clean_taxi_df.withColumn(
    "row_number",
    row_number().over(fare_window)
)

row_number_df.select(
    "VendorID",
    "fare_amount",
    "row_number"
).show(10)

+--------+-----------+----------+
|VendorID|fare_amount|row_number|
+--------+-----------+----------+
|       2|     2221.3|         1|
|       2|     1616.5|         2|
|       2|      912.3|         3|
|       2|      899.0|         4|
|       2|      820.0|         5|
|       2|      761.1|         6|
|       2|      749.2|         7|
|       2|      744.3|         8|
|       2|      739.4|         9|
|       2|      700.0|        10|
+--------+-----------+----------+
only showing top 10 rows


In [65]:
rank_result_df = clean_taxi_df.withColumn(
    "rank",
    rank().over(fare_window)
)

rank_result_df.select(
    "VendorID",
    "fare_amount",
    "rank"
).show(10)

+--------+-----------+----+
|VendorID|fare_amount|rank|
+--------+-----------+----+
|       2|     2221.3|   1|
|       2|     1616.5|   2|
|       2|      912.3|   3|
|       2|      899.0|   4|
|       2|      820.0|   5|
|       2|      761.1|   6|
|       2|      749.2|   7|
|       2|      744.3|   8|
|       2|      739.4|   9|
|       2|      700.0|  10|
+--------+-----------+----+
only showing top 10 rows


In [66]:
dense_rank_result_df = clean_taxi_df.withColumn(
    "dense_rank",
    dense_rank().over(fare_window)
)

dense_rank_result_df.select(
    "VendorID",
    "fare_amount",
    "dense_rank"
).show(10)

+--------+-----------+----------+
|VendorID|fare_amount|dense_rank|
+--------+-----------+----------+
|       2|     2221.3|         1|
|       2|     1616.5|         2|
|       2|      912.3|         3|
|       2|      899.0|         4|
|       2|      820.0|         5|
|       2|      761.1|         6|
|       2|      749.2|         7|
|       2|      744.3|         8|
|       2|      739.4|         9|
|       2|      700.0|        10|
+--------+-----------+----------+
only showing top 10 rows


#PART 8 — PERFORMANCE OPTIMIZATION

In [67]:
import time

In [68]:
start_time_before_cache = time.time()

clean_taxi_df.groupBy(
    "payment_type"
).count().show()

end_time_before_cache = time.time()

execution_time_before_cache = (
    end_time_before_cache -
    start_time_before_cache
)

print(
    "Execution Time Before Caching:",
    execution_time_before_cache,
    "seconds"
)

+------------+-------+
|payment_type|  count|
+------------+-------+
|           1|2298422|
|           3|  10649|
|           2| 422945|
|           4|  22879|
+------------+-------+

Execution Time Before Caching: 21.31794548034668 seconds


In [69]:
clean_taxi_df.cache()

# Materialize the cache
clean_taxi_df.count()

print("Dataset Cached Successfully!")

Dataset Cached Successfully!


In [70]:
start_time_after_cache = time.time()

clean_taxi_df.groupBy(
    "payment_type"
).count().show()

end_time_after_cache = time.time()

execution_time_after_cache = (
    end_time_after_cache -
    start_time_after_cache
)

print(
    "Execution Time After Caching:",
    execution_time_after_cache,
    "seconds"
)

+------------+-------+
|payment_type|  count|
+------------+-------+
|           1|2298422|
|           3|  10649|
|           2| 422945|
|           4|  22879|
+------------+-------+

Execution Time After Caching: 4.6216442584991455 seconds


In [71]:
print("========== PERFORMANCE COMPARISON ==========")

print(
    "Before Caching:",
    execution_time_before_cache,
    "seconds"
)

print(
    "After Caching:",
    execution_time_after_cache,
    "seconds"
)

if execution_time_after_cache < execution_time_before_cache:
    print("Caching improved execution time.")
else:
    print(
        "Caching did not improve execution time in this run."
    )

========== PERFORMANCE COMPARISON ==========
Before Caching: 21.31794548034668 seconds
After Caching: 4.6216442584991455 seconds
Caching improved execution time.


In [72]:
print(
    "Partitions Before:",
    clean_taxi_df.rdd.getNumPartitions()
)

optimized_taxi_df = clean_taxi_df.repartition(4)

print(
    "Partitions After:",
    optimized_taxi_df.rdd.getNumPartitions()
)

Partitions Before: 200
Partitions After: 4


In [73]:
clean_taxi_df.groupBy(
    "payment_type"
).count().explain(True)

== Parsed Logical Plan ==
'Aggregate ['payment_type], ['payment_type, 'count(1) AS count#3095]
+- Filter atleastnnonnulls(19, VendorID#0, tpep_pickup_datetime#1, tpep_dropoff_datetime#2, passenger_count#3L, trip_distance#4, RatecodeID#5L, store_and_fwd_flag#6, PULocationID#7, DOLocationID#8, payment_type#9L, fare_amount#10, extra#11, mta_tax#12, tip_amount#13, tolls_amount#14, improvement_surcharge#15, total_amount#16, congestion_surcharge#17, Airport_fee#18)
   +- Filter (fare_amount#10 >= cast(0 as double))
      +- Filter (trip_distance#4 > cast(0 as double))
         +- Deduplicate [DOLocationID#8, improvement_surcharge#15, tpep_dropoff_datetime#2, PULocationID#7, tolls_amount#14, tip_amount#13, passenger_count#3L, store_and_fwd_flag#6, extra#11, congestion_surcharge#17, total_amount#16, tpep_pickup_datetime#1, mta_tax#12, trip_distance#4, Airport_fee#18, RatecodeID#5L, VendorID#0, payment_type#9L, fare_amount#10]
            +- Relation [VendorID#0,tpep_pickup_datetime#1,tpep_drop

#PART 9 — SPARK UI

In [74]:
!pip install -q pyngrok

In [75]:
from pyngrok import ngrok
ngrok.set_auth_token("3HMQSImSmPUEeOt69nhWMnmFEhU_5UzFpqqE7ns5E4LkMtnSJ")

In [77]:
public_spark_ui = ngrok.connect(4040)

print("Public Spark UI:")
print(public_spark_ui)

Public Spark UI:
NgrokTunnel: "https://delegator-wasp-freedom.ngrok-free.dev" -> "http://localhost:4040"


In [78]:
clean_taxi_df.count()

2754895

In [79]:
clean_taxi_df.groupBy(
    "payment_type"
).count().show()

+------------+-------+
|payment_type|  count|
+------------+-------+
|           1|2298422|
|           3|  10649|
|           2| 422945|
|           4|  22879|
+------------+-------+



In [80]:
clean_taxi_df.orderBy(
    "fare_amount"
).show(10)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|       2| 2024-01-05 23:55:26|  2024-01-06 00:10:51|              1|         2.78|         1|                 N|         100|         145|           2|        0.0|  0.0|    0.5|       0.

In [81]:
import platform
import sys
from datetime import datetime

execution_log = f"""
Execution Log
========================================

Student Name : Muhammad Fahadullah
Roll No      : 231980071
Date         : {datetime.now().strftime('%d-%m-%Y %H:%M:%S')}

Environment
----------------------------------------
Platform           : Google Colab
Operating System   : {platform.system()}
Python Version     : {sys.version.split()[0]}
Apache Spark       : {spark_session.version}
Spark Master       : {spark_session.sparkContext.master}

Assignment Parts
----------------------------------------
[✓] Part 1 - Environment Setup
[✓] Part 2 - Data Loading
[✓] Part 3 - Exploratory Data Analysis
[✓] Part 4 - Data Cleaning
[✓] Part 5 - Spark Transformations
[✓] Part 6 - Spark SQL
[✓] Part 7 - Window Functions
[✓] Part 8 - Performance Optimization
[✓] Part 9 - Spark UI Analysis

Dataset Statistics
----------------------------------------
Total Records : {clean_taxi_df.count()}
Total Columns : {len(clean_taxi_df.columns)}

Status
----------------------------------------
Execution Completed Successfully.
"""

with open("execution_log.txt", "w") as log_file:
    log_file.write(execution_log)

print(execution_log)
print("execution_log.txt created successfully!")


Execution Log

Student Name : Muhammad Fahadullah
Roll No      : 231980071
Date         : 03-08-2026 21:02:48

Environment
----------------------------------------
Platform           : Google Colab
Operating System   : Linux
Python Version     : 3.12.13
Apache Spark       : 4.0.3
Spark Master       : local[*]

Assignment Parts
----------------------------------------
[✓] Part 1 - Environment Setup
[✓] Part 2 - Data Loading
[✓] Part 3 - Exploratory Data Analysis
[✓] Part 4 - Data Cleaning
[✓] Part 5 - Spark Transformations
[✓] Part 6 - Spark SQL
[✓] Part 7 - Window Functions
[✓] Part 8 - Performance Optimization
[✓] Part 9 - Spark UI Analysis

Dataset Statistics
----------------------------------------
Total Records : 2754895
Total Columns : 19

Status
----------------------------------------
Execution Completed Successfully.

execution_log.txt created successfully!
